# RAG Pipeline - Exp6

* About
  * Save embeddings in Railway with non-llamaindex embeddings and use it in later RAG pipeline
  * Couldn't use LlamaIndex to upload generated embeddings into Railway pgvector

In [1]:
%load_ext autoreload
%autoreload 2

import os
import yaml
os.environ.pop("OPENAI_API_KEY", None)

from datasets import load_dataset
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

### Before Embedding

In [6]:
fiqa_eval = load_dataset("explodinggradients/fiqa", "ragas_eval")['baseline']

output_dir = "fiqa_raw_text"
os.makedirs(output_dir, exist_ok=True)

rag_lst = []
documents = []  # to store documents for llamindex
for idx, record in enumerate(fiqa_eval):
    context = ''.join(record['contexts'])
    gt = ''.join(record['ground_truths'])
    if 'answer' in record.keys():
        ai0_answer = record['answer'].strip()
    else:
        ai0_answer = None

    with open(os.path.join(output_dir, f"{idx}.txt"), "w", encoding="utf-8") as f:
        f.write(context)
    rag_lst.append({
        'question': record['question'],
        'context': context,
        'context_ct': len(record['contexts']),
        'ground_truth': gt,
        'ai0_answer': ai0_answer
    })
    documents.append(context)


rag_df = pd.DataFrame(rag_lst)
print(rag_df.shape, max(rag_df['context_ct']), min(rag_df['context_ct']))
print(len(documents))
rag_df.head()

(30, 5) 1 1
30


,question,context,context_ct,ground_truth,ai0_answer
0,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...,1,Have the check reissued to the proper payee.Ju...,The best way to deposit a cheque issued to an ...
1,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,1,Sure you can. You can fill in whatever you wa...,"Yes, you can send a money order from USPS as a..."
2,1 EIN doing business under multiple business n...,You're confusing a lot of things here. Company...,1,You're confusing a lot of things here. Company...,"Yes, it is possible to have one EIN doing busi..."
3,Applying for and receiving business credit,Set up a meeting with the bank that handles yo...,1,"""I'm afraid the great myth of limited liabilit...",Applying for and receiving business credit can...
4,401k Transfer After Business Closure,The time horizon for your 401K/IRA is essentia...,1,You should probably consult an attorney. Howev...,If your employer has closed and you need to tr...


In [7]:
with open("all_rag_pipeline_config.yaml", "r", encoding="utf-8") as f:
     all_config= yaml.safe_load(f)

### Embedding

#### create embeddings

* On Local Machine
  * Install DBeaver: https://dbeaver.io/download/
* Railway Setup
  * Railway create a service connect to Github Repo for backend
  * Create --> Template --> Type "pgvector" --> Choose the one with most downloads (psql 18)
    * Don't use the "PgVector" for RAG, that one doesn't have `host` and `password` in public database url, can't upload data from here
  * Connnect Railway service and pgvector by creating a new variable in the service --> `Add Reference` --> `DATABASE_URL`

In [12]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "BAAI/bge-small-en-v1.5"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_embeddings_batch(texts, batch_size=32):
    embeddings_list = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt")
        with torch.no_grad():
            outputs = model(**inputs)
            batch_embeddings = outputs.last_hidden_state.mean(dim=1)
            embeddings_list.extend(batch_embeddings.tolist())
    return embeddings_list


embeddings = get_embeddings_batch(documents, batch_size=16)

In [ ]:
import psycopg2


DATABASE_URL_PUBLIC = os.getenv("DATABASE_URL_PUBLIC_RAG_DR")

conn = psycopg2.connect(DATABASE_URL_PUBLIC)
cur = conn.cursor()

cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
conn.commit()

cur.execute("DROP TABLE IF EXISTS rag_dr_embeddings;")
cur.execute("""
CREATE TABLE IF NOT EXISTS rag_dr_embeddings (
    context TEXT,
    embedding vector(384)  -- match BAAI/bge-small-en-v1.5
);
""")
conn.commit()

In [18]:
args_str = ",".join(
    cur.mogrify("(%s, %s)", (context, emb)).decode("utf-8")
    for context, emb in zip(documents, embeddings)
)

cur.execute(f"INSERT INTO rag_dr_embeddings (context, embedding) VALUES {args_str}")
conn.commit()

cur.close()
conn.close()

In [ ]:
cur.execute("SELECT current_database();")
print("Connected to database:", cur.fetchone())

cur.execute("""
SELECT table_name FROM information_schema.tables 
WHERE table_schema='public';
""")
print("Tables:", cur.fetchall())

conn.close()

Connected to database: ('railway',)
Tables: [('rag_dr_embeddings',)]
